In [10]:
import asyncio
import os

from dotenv import load_dotenv
from google import genai
from google.genai import types

# ---------------------------------------------------
# Load Environment Variables
# ---------------------------------------------------

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# ---------------------------------------------------
# Gemini Client
# ---------------------------------------------------

client = genai.Client(api_key=GEMINI_API_KEY)


AI agent Architectural Flow 
```
User Query
   ↓
LLM decides
   ↓
┌─────────────────────────────┐
│ State                       │
│ decision                    │
│ tool_required               │
│ tool_name                   │
│ tool_params                 │
│ final_ans                   │
└─────────────────────────────┘
   ↓
tool_required?
 ├── No → final_ans → STOP
 │
 └── Yes
      ↓
   Missing parameters?
      ├── Yes → ask user → update state → LLM again
      │
      └── No → execute Python tool
                    ↓
               tool result
                    ↓
                LLM again
                    ↓
             final answer → STOP

```

In [ ]:
import json
from typing import Any, Callable, Dict, Optional

import asyncio
import os

from dotenv import load_dotenv
from google import genai
from google.genai import types
load_dotenv()

True

In [ ]:

# ============================================================
# 1. LLM CLIENT
# ============================================================


GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
client = genai.Client()
# client = genai.Client(api_key="GEMINI_API_KEY")



In [13]:
response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents="Explain what an AI agent is in one sentence."
)

print(response.text)

An AI agent is a software program capable of perceiving its environment, reasoning through information, and autonomously taking actions to achieve specific goals.


In [14]:
# ============================================================
# 2. AGENT STATE
# ============================================================

class AgentState:

    def __init__(self, user_query: str):

        self.user_query = user_query
        self.decision: str = ""
        self.tool_required: bool = False
        self.tool_name: Optional[str] = None
        self.tool_params: Dict[str, Any] = {}
        self.missing_params: list[str] = []
        self.tool_result: Optional[Any] = None
        self.final_ans: Optional[str] = None
        # Gemini conversation history
        self.messages: list[types.Content] = []


In [15]:
# ============================================================
# 3. TOOLS
# ============================================================

def get_weather(city: str, days: int):
    """
    Example tool.
    Replace this with your actual Python function/API.
    """

    return {
        "city": city,
        "days": days,
        "temperature": "28°C",
        "condition": "Partly cloudy",
        "forecast": f"Weather forecast for {days} days in {city}"
    }


def calculate_sum(a: float, b: float):
    """
    Another example tool.
    """

    return {
        "a": a,
        "b": b,
        "result": a + b
    }

In [16]:
# ============================================================
# 4. TOOL REGISTRY
# ============================================================

TOOLS = {
    "get_weather": {
        "function": get_weather,
        "description": "Get weather information for a city.",
        "parameters": {
            "city": {
                "type": "string",
                "description": "Name of the city"
            },
            "days": {
                "type": "integer",
                "description": "Number of forecast days"
            }
        },
        "required": ["city", "days"]
    },

    "calculate_sum": {
        "function": calculate_sum,
        "description": "Calculate the sum of two numbers.",
        "parameters": {
            "a": {
                "type": "number",
                "description": "First number"
            },
            "b": {
                "type": "number",
                "description": "Second number"
            }
        },
        "required": ["a", "b"]
    }
}

In [17]:
# ============================================================
# 5. DYNAMIC TOOL DESCRIPTION
# ============================================================

def build_tool_description(
    tools: Dict[str, Dict[str, Any]]
) -> str:

    descriptions = []

    for name, tool in tools.items():

        parameter_lines = []

        for param_name, param_info in tool["parameters"].items():

            required = (
                param_name in tool["required"]
            )

            required_text = (
                "required"
                if required
                else "optional"
            )

            parameter_lines.append(
                f"- {param_name}: "
                f"{param_info['type']}, "
                f"{required_text}. "
                f"{param_info['description']}"
            )

        descriptions.append(
            f"""
Tool name: {name}

Description:
{tool['description']}

Parameters:
{chr(10).join(parameter_lines)}
"""
        )

    return "\n".join(descriptions)


TOOL_DESCRIPTION = build_tool_description(TOOLS)

In [18]:
print(TOOL_DESCRIPTION)


Tool name: get_weather

Description:
Get weather information for a city.

Parameters:
- city: string, required. Name of the city
- days: integer, required. Number of forecast days


Tool name: calculate_sum

Description:
Calculate the sum of two numbers.

Parameters:
- a: number, required. First number
- b: number, required. Second number



In [ ]:

SYSTEM_PROMPT = f"""
You are an intelligent tool-using agent.

Your job is to analyze the user's request and decide whether
a registered Python tool/function is required.

You MUST return ONLY valid JSON.

Do not return Markdown.
Do not use code fences.
Do not add explanations outside JSON.

REQUIRED JSON STRUCTURE:

{{
    "decision": "short explanation of your decision",
    "tool_required": true,
    "tool_name": "tool name or null",
    "tool_params": {{}},
    "final_ans": "answer or null"
}}

AVAILABLE TOOLS:

{TOOL_DESCRIPTION}


========================
IMPORTANT AGENT RULES
========================

RULE 1: NO TOOL REQUIRED

If the user's request can be answered without using a tool:

- tool_required must be false.
- tool_name must be null.
- tool_params must be an empty object.
- final_ans must contain the final answer.

Example:

User:
"Hello"

Response:

{{
    "decision": "The user is greeting the agent and no tool is required.",
    "tool_required": false,
    "tool_name": null,
    "tool_params": {{}},
    "final_ans": "Hello! How can I help you today?"
}}


RULE 2: TOOL REQUIRED

If the user's request requires a tool:

- tool_required MUST be true.
- tool_name MUST be one of the registered tools.
- Extract every parameter that the user has already provided.
- Do not invent missing parameter values.
- tool_params must contain only known parameters.
- final_ans must be null.

IMPORTANT:

A tool can be required even when some parameters are missing.

NEVER change tool_required to false just because
some required parameters are missing.

The Python runtime will determine which parameters are missing.

Example:

User:
"I wanna add two numbers"

Correct response:

{{
    "decision": "The calculate_sum tool is required, but the numbers have not been provided yet.",
    "tool_required": true,
    "tool_name": "calculate_sum",
    "tool_params": {{}},
    "final_ans": null
}}

The Python runtime will calculate:

missing_params = ["a", "b"]


Another example:

User:
"What is the weather in Pune?"

If the weather tool requires both city and days:

{{
    "decision": "The weather tool is required. The city is known but the forecast duration has not been provided.",
    "tool_required": true,
    "tool_name": "get_weather",
    "tool_params": {{
        "city": "Pune"
    }},
    "final_ans": null
}}

The Python runtime will calculate:

missing_params = ["days"]


RULE 3: NEVER INVENT PARAMETERS

If the user has not provided a parameter, do not guess it.

For example:

User:
"Calculate the sum of 10 and 20."

Response:

{{
    "decision": "The calculate_sum tool is required and both numbers are available.",
    "tool_required": true,
    "tool_name": "calculate_sum",
    "tool_params": {{
        "a": 10,
        "b": 20
    }},
    "final_ans": null
}}

If the user says:

"Calculate the sum of 10"

Response:

{{
    "decision": "The calculate_sum tool is required, but the second number is missing.",
    "tool_required": true,
    "tool_name": "calculate_sum",
    "tool_params": {{
        "a": 10
    }},
    "final_ans": null
}}

The Python runtime will calculate:

missing_params = ["b"]


RULE 4: UPDATE PARAMETERS

If the user provides a missing parameter in a later message,
combine it with the parameters already collected.

Example:

Previously collected:

{{
    "city": "Pune"
}}

User:
"For 5 days."

Response:

{{
    "decision": "The forecast duration has been provided and the weather tool is ready to execute.",
    "tool_required": true,
    "tool_name": "get_weather",
    "tool_params": {{
        "city": "Pune",
        "days": 5
    }},
    "final_ans": null
}}

Do not discard parameters that were already collected.

Do not invent parameters that the user has not provided.


RULE 5: AFTER TOOL EXECUTION

The Python runtime will provide the result of the executed tool.

Use that result to formulate the final answer.

After receiving a successful tool result:

- tool_required must be false.
- tool_name must be null.
- tool_params must be an empty object.
- final_ans must contain the natural-language answer.

Do not execute the same tool again merely because a result was provided.

Example:

Tool result:

{{
    "city": "Pune",
    "temperature": "28°C",
    "condition": "Sunny"
}}

Response:

{{
    "decision": "The weather tool returned the requested information.",
    "tool_required": false,
    "tool_name": null,
    "tool_params": {{}},
    "final_ans": "The weather in Pune is sunny with a temperature of 28°C."
}}


RULE 6: PYTHON OWNS MISSING PARAMETER DETECTION

Python is responsible for determining which required
parameters are missing.

The LLM must NOT calculate missing_params.

The LLM must only:

1. Identify whether a tool is required.
2. Identify which tool is required.
3. Extract parameters already provided by the user.
4. Preserve previously collected parameters.
5. Leave missing parameters out of tool_params.

Python will then calculate:

missing_params = get_missing_parameters(
    tool_name,
    tool_params
)


RULE 7: CLARIFICATION QUESTIONS

When Python detects missing parameters, Python will send
an internal instruction asking you to generate a clarification
question.

In this situation:

- tool_required must be false.
- tool_name must be null.
- tool_params must be an empty object.
- final_ans must contain ONLY the clarification question.

Example:

Internal instruction:

"The calculate_sum tool requires parameters ['a', 'b'].
Ask the user to provide them."

Response:

{{
    "decision": "The required parameters are missing and the user needs to provide them.",
    "tool_required": false,
    "tool_name": null,
    "tool_params": {{}},
    "final_ans": "Could you please provide the two numbers you would like me to add?"
}}

IMPORTANT:

This is ONLY for the clarification response generated after
Python explicitly asks for a clarification question.

Do NOT use this rule during the initial analysis of the user's request.

During the initial analysis, if a tool is required,
tool_required MUST remain true even when parameters are missing.


RULE 8: JSON VALIDITY

Always return valid JSON.

Use double quotes around JSON keys and string values.

Do not include trailing commas.

Return exactly these five keys:

- decision
- tool_required
- tool_name
- tool_params
- final_ans
"""

In [20]:
# ============================================================
# 7. GEMINI LLM CALL
# ============================================================

MODEL = "gemini-3.1-flash-lite"

def call_llm(
    messages: list[types.Content]
) -> Dict[str, Any]:

    """
    Sends the conversation to Gemini and parses its JSON response.
    """

    response = client.models.generate_content(
        model=MODEL,
        contents=messages,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=0,
            response_mime_type="application/json"
        )
    )

    content = response.text

    if not content:
        raise ValueError(
            "Gemini returned an empty response."
        )

    try:
        result = json.loads(content)

    except json.JSONDecodeError as error:

        raise ValueError(
            f"Gemini returned invalid JSON:\n{content}"
        ) from error

    return result

In [21]:

# ============================================================
# 8. VALIDATE LLM STATE
# ============================================================

def validate_llm_state(
    result: Dict[str, Any]
) -> None:

    required_keys = {
        "decision",
        "tool_required",
        "tool_name",
        "tool_params",
        "final_ans"
    }

    missing_keys = (
        required_keys - result.keys()
    )

    if missing_keys:
        raise ValueError(
            f"Missing keys from LLM response: {missing_keys}"
        )

    if not isinstance(
        result["tool_required"],
        bool
    ):
        raise ValueError(
            "tool_required must be a boolean."
        )

    if not isinstance(
        result["tool_params"],
        dict
    ):
        raise ValueError(
            "tool_params must be a dictionary."
        )

    if result["tool_required"]:

        if not result["tool_name"]:
            raise ValueError(
                "tool_name is required when tool_required is true."
            )

        if result["final_ans"] is not None:
            raise ValueError(
                "final_ans must be null when a tool is required."
            )

    else:

        if result["tool_name"] is not None:
            raise ValueError(
                "tool_name must be null when no tool is required."
            )


In [22]:
# ============================================================
# 9. MISSING PARAMETER DETECTION
# ============================================================

def get_missing_parameters(
    tool_name: str,
    collected_params: Dict[str, Any]
) -> list[str]:

    tool = TOOLS[tool_name]

    missing = []

    for param in tool["required"]:

        if (
            param not in collected_params
            or collected_params[param] is None
        ):
            missing.append(param)

    return missing


In [23]:
get_missing_parameters("get_weather",{'city':'pune'})

['days']

In [25]:
# ask_for_parameters("get_weather",['days'])

In [ ]:

# ============================================================
# 9. PARAMETER TYPE CONVERSION
# ============================================================

def convert_parameter(
    value: str,
    param_type: str
) -> Any:

    value = value.strip()

    if param_type == "integer":
        return int(value)

    if param_type == "number":
        return float(value)

    if param_type == "boolean":
        value_lower = value.lower()

        if value_lower in ["true", "yes", "y"]:
            return True

        if value_lower in ["false", "no", "n"]:
            return False

        raise ValueError(
            "Please enter true/false."
        )

    return value

In [ ]:
# ============================================================
# 10. EXECUTE PYTHON TOOL
# ============================================================

def execute_tool(
    tool_name: str,
    tool_params: Dict[str, Any]
) -> Any:

    if tool_name not in TOOLS:
        raise ValueError(
            f"Unknown tool: {tool_name}"
        )

    tool = TOOLS[tool_name]

    function: Callable = tool["function"]

    print("\n[TOOL CALL]")
    print(f"Name: {tool_name}")
    print(
        f"Parameters: "
        f"{json.dumps(tool_params)}"
    )

    result = function(**tool_params)

    print("\n[TOOL RESULT]")
    print(json.dumps(result, indent=2))

    return result

In [29]:
tool_name = 'get_weather'
tool_params = {
    'city':'Pune',
    'days':'4'
}
execute_tool(tool_name,tool_params)


[TOOL CALL]
Name: get_weather
Parameters: {"city": "Pune", "days": "4"}

[TOOL RESULT]
{
  "city": "Pune",
  "days": "4",
  "temperature": "28\u00b0C",
  "condition": "Partly cloudy",
  "forecast": "Weather forecast for 4 days in Pune"
}


{'city': 'Pune',
 'days': '4',
 'temperature': '28°C',
 'condition': 'Partly cloudy',
 'forecast': 'Weather forecast for 4 days in Pune'}

In [ ]:
# ============================================================
# 11. PRINT STATE
# ============================================================

def print_state(state: AgentState):

    print("\n========== AGENT STATE ==========")

    print(
        json.dumps(
            {
                "decision": state.decision,
                "tool_required": state.tool_required,
                "tool_name": state.tool_name,
                "tool_params": state.tool_params,
                "missing_params": state.missing_params,
                "tool_result": state.tool_result,
                "final_ans": state.final_ans
            },
            indent=2,
            default=str
        )
    )

    print("=================================\n")


In [ ]:

# ============================================================
# 12. MAIN AGENT LOOP
# ============================================================

def run_agent_turn(state: AgentState) -> str:
    """
    Executes one agent turn using state.messages history.

    Gemini can be called multiple times internally:
    1. Decide whether a tool is required.
    2. If parameters are missing, generate a clarification question.
    3. After tool execution, generate the final answer.
    """

    max_iterations = 10
    iteration = 0

    while iteration < max_iterations:
        iteration += 1

        print(f"\n========== ITERATION {iteration} ==========")

        # ============================================================
        # STEP 1: CALL GEMINI
        # ============================================================

        result = call_llm(state.messages)

        # ============================================================
        # STEP 2: VALIDATE GEMINI OUTPUT
        # ============================================================

        validate_llm_state(result)

        # ============================================================
        # STEP 3: SYNCHRONIZE LLM STATE
        # ============================================================

        state.decision = result.get("decision")
        state.tool_required = result.get("tool_required", False)
        state.tool_name = result.get("tool_name")
        state.tool_params = result.get("tool_params", {})
        state.final_ans = result.get("final_ans")

        # ============================================================
        # STEP 4: COMPUTE MISSING PARAMETERS
        # Python is responsible for this.
        # ============================================================

        if state.tool_required and state.tool_name in TOOLS:

            state.missing_params = get_missing_parameters(
                state.tool_name,
                state.tool_params
            )

        else:
            state.missing_params = []

        # ============================================================
        # STEP 5: PRINT CURRENT STATE
        # ============================================================

        print_state(state)

        # ============================================================
        # STEP 6: NO TOOL REQUIRED
        # ============================================================

        if not state.tool_required:

            if not state.final_ans:
                raise ValueError(
                    "Gemini returned no final answer."
                )

            # Save Gemini response to conversation history
            state.messages.append(
                types.Content(
                    role="model",
                    parts=[
                        types.Part(
                            text=json.dumps(result)
                        )
                    ]
                )
            )

            print("\nFINAL ANSWER:")
            print(state.final_ans)

            return state.final_ans

        # ============================================================
        # STEP 7: VALIDATE TOOL
        # ============================================================

        if state.tool_name not in TOOLS:

            raise ValueError(
                f"Gemini selected an invalid tool: {state.tool_name}"
            )

        # ============================================================
        # STEP 8: MISSING PARAMETERS
        # ============================================================

        if state.missing_params:

            print(
                f"\nMissing parameters detected by runtime: "
                f"{state.missing_params}"
            )

            # Save the original tool decision
            state.messages.append(
                types.Content(
                    role="model",
                    parts=[
                        types.Part(
                            text=json.dumps(result)
                        )
                    ]
                )
            )

            # Ask Gemini to generate clarification question
            state.messages.append(
                types.Content(
                    role="user",
                    parts=[
                        types.Part(
                            text=(
                                f"The tool '{state.tool_name}' requires "
                                f"parameter(s) {state.missing_params} "
                                "which are currently missing.\n"
                                "Set tool_required to false and ask the user "
                                "to provide these values in final_ans."
                            )
                        )
                    ]
                )
            )

            # ------------------------------------------------------------
            # CALL GEMINI ONLY FOR CLARIFICATION
            # ------------------------------------------------------------

            clarification_result = call_llm(state.messages)

            validate_llm_state(clarification_result)

            clarification_question = clarification_result.get("final_ans")

            # Save Gemini clarification response to history
            state.messages.append(
                types.Content(
                    role="model",
                    parts=[
                        types.Part(
                            text=json.dumps(clarification_result)
                        )
                    ]
                )
            )

            print("\nCLARIFICATION:")
            print(clarification_question)

            return clarification_question

        # ============================================================
        # STEP 9: EXECUTE TOOL
        # ============================================================

        state.tool_result = execute_tool(
            state.tool_name,
            state.tool_params
        )

        print("\n[TOOL CALL]")
        print(f"Name: {state.tool_name}")
        print(
            f"Parameters: "
            f"{json.dumps(state.tool_params)}"
        )

        print("\n[TOOL RESULT]")
        print(
            json.dumps(
                state.tool_result,
                indent=2,
                ensure_ascii=False
            )
        )

        # ============================================================
        # STEP 10: SAVE TOOL CALL TO HISTORY
        # ============================================================

        state.messages.append(
            types.Content(
                role="model",
                parts=[
                    types.Part(
                        text=json.dumps(result)
                    )
                ]
            )
        )

        # ============================================================
        # STEP 11: SEND TOOL RESULT BACK TO GEMINI
        # ============================================================

        tool_result_message = (
            f"Tool executed successfully.\n\n"
            f"Tool name: {state.tool_name}\n"
            f"Tool parameters: "
            f"{json.dumps(state.tool_params)}\n"
            f"Tool result: "
            f"{json.dumps(state.tool_result)}\n\n"
            "Generate the final natural-language answer using "
            "the tool result."
        )

        state.messages.append(
            types.Content(
                role="user",
                parts=[
                    types.Part(
                        text=tool_result_message
                    )
                ]
            )
        )

        # Continue loop.
        # Gemini will generate final answer.
        continue

    raise RuntimeError(
        f"Agent exceeded maximum iterations ({max_iterations})."
    )



In [ ]:

# ============================================================
# APPLICATION ENTRY POINT
# ============================================================

if __name__ == "__main__":

    print("Gemini Tool-Calling Agent")
    print("Type 'exit' to quit.\n")

    # Persistent state for the entire conversation
    session_state = AgentState(
        user_query=""
    )

    while True:

        user_query = input("\nUser: ").strip()

        if user_query.lower() == "exit":
            print("Agent stopped.")
            break

        if not user_query:
            continue

        try:

            # ========================================================
            # IMPORTANT:
            # DO NOT RESET tool_name/tool_params here.
            #
            # They may contain a pending tool call waiting for
            # missing parameters.
            # ========================================================

            session_state.user_query = user_query

            # --------------------------------------------------------
            # Add actual user message to conversation
            # --------------------------------------------------------

            session_state.messages.append(
                types.Content(
                    role="user",
                    parts=[
                        types.Part(
                            text=user_query
                        )
                    ]
                )
            )

            # --------------------------------------------------------
            # Execute agent turn
            # --------------------------------------------------------

            run_agent_turn(session_state)

        except Exception as error:

            print(
                f"\nAgent error: {error}\n"
            )

Gemini Tool-Calling Agent
Type 'exit' to quit.



========== ITERATION 1 ==========

Agent error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


========== ITERATION 1 ==========

Agent error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Agent stopped.
